# Method C: Slow & Steady Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/investigate-oasis-sheets-aYOlG/notebooks/method-c-slow-validation.ipynb)

## Theory

The previous bulk download (0.3s delay) returned correct historical data only
in short clusters. Hypothesis: **going slower prevents the collapse** to
current content.

## Approach

1. Start with a **small sample** — mix of known-correct and known-broken revisions
2. Use **longer delays** between API calls (configurable, default 5s)
3. **Canary detection**: if we get the same hash N times in a row, stop
4. **Compare** against previous run's data on Drive
5. **Replace** files on Drive when we get genuinely different (better) content
6. Write **structured progress** to Drive after every single request

## Known State from Previous Run

| Sheet | "Current" Hash | Current Size | Historical Sizes |
|-------|---------------|-------------|------------------|
| Library | `716eacf3...` / `7bd3c0e4...` | 640,702 | 596K-607K |
| Documents | `d1301ded...` / `5fd91297...` | 931,907 / 931,904 | 790K-910K |

In [ ]:
# === Step 0: Auth + Mount ===
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import google.auth
from google.auth.transport.requests import Request as AuthRequest
creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive: {DRIVE_DIR}')

## Step 1: Test Plan Configuration

**Edit this cell** to adjust what gets tested. Each entry is a revision
number with its expected behavior from the previous run.

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  CONFIGURATION — adjust these for each run                     │
# └─────────────────────────────────────────────────────────────────┘

# Delay between API calls (seconds). Adaptive backoff adjusts this.
REQUEST_DELAY = 5.0

# Stop after this many consecutive identical content hashes.
CANARY_THRESHOLD = 10

# Which sheet to download
TEST_SHEET = 'ubl25_library'
# TEST_SHEET = 'ubl25_documents'

# Max revision number per sheet (from Revisions API discovery).
# Set to 0 to stop on first 404 (auto-detect end).
MAX_REVS = {
    'ubl25_library':   2005,
    'ubl25_documents': 2204,
}

MAX_REV = MAX_REVS[TEST_SHEET]

# ── Build sequential test plan: every revision from 1 to MAX_REV ──
TEST_PLAN = [(rev, 'all') for rev in range(1, MAX_REV + 1)]

print(f'Test plan: {TEST_SHEET}')
print(f'  {len(TEST_PLAN)} revisions (rev 1 -> {MAX_REV})')
print(f'  Delay: {REQUEST_DELAY}s between requests (adaptive)')
print(f'  Canary: stop after {CANARY_THRESHOLD} identical hashes in a row')
est_minutes = len(TEST_PLAN) * REQUEST_DELAY / 60
print(f'  Estimated runtime: ~{est_minutes:.0f} min ({est_minutes/60:.1f} hrs) at initial rate')

In [ ]:
# === Step 2: Helpers ===
import json, time, hashlib, gzip, zipfile, io, re
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError

SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

# Known "current content" hashes from the previous run.
# If we see these, the API returned the latest version (not historical).
CURRENT_HASHES = {
    'ubl25_library': {
        '716eacf3ebe60d7622e4c9c439e3807b7cf035cd8581ad62c495b8d154064010',
        '7bd3c0e40778115f96c0e5515c093e23e15d1c7d3bbe8b34f29c80f5b97de823',
    },
    'ubl25_documents': {
        'd1301dedea016c962d0dbdb8f287bdd0f60c6911086e5f29f0bca389afa08ac3',
        '5fd9129713bef9336a68ea19821ab406e0e1d6317b1d4a2d7a5ca2aba8a7f8e7',
    },
}

CURRENT_SIZES = {
    'ubl25_library': 640702,
    'ubl25_documents': 931907,
}


def now_iso():
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')


def export_revision_ods(sheet_id, rev_num, retry_wait=9.0):
    """Method C: download a specific revision as ODS.
    Returns (ods_bytes, http_status, error_msg, hit_429).
    On 429: waits retry_wait seconds then retries (up to 2 retries)."""
    url = (f'https://docs.google.com/spreadsheets/export'
           f'?id={sheet_id}&revision={rev_num}&exportFormat=ods')
    headers = {'Authorization': f'Bearer {TOKEN}'}
    hit_429 = False
    for attempt in range(3):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                data = resp.read()
                if len(data) > 500:
                    return data, 200, None, hit_429
                return None, 200, f'too small ({len(data)} bytes)', hit_429
        except HTTPError as e:
            if e.code == 429 and attempt < 2:
                hit_429 = True
                print(f'429({retry_wait:.0f}s)...', end='', flush=True)
                time.sleep(retry_wait)
                continue
            if e.code in (500, 502, 503) and attempt < 2:
                print(f'retry({e.code}, {retry_wait:.0f}s)...', end='', flush=True)
                time.sleep(retry_wait)
                continue
            return None, e.code, str(e.code), hit_429
        except Exception as exc:
            if attempt < 2:
                time.sleep(retry_wait)
                continue
            return None, 0, str(exc), hit_429
    return None, 0, 'max retries', hit_429


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return None


print('Helpers ready')

In [ ]:
# === Step 3: Load manifest + spot-check against actual Drive files ===
#
# The manifest has content_hash per revision, but let's VERIFY those
# are correct by reading the actual .ods.gz files from Drive and
# computing the hash ourselves.

manifest_path = DRIVE_DIR / f'manifest-{TEST_SHEET}.json'
prev_data = {}  # rev_num -> {content_hash, ods_size, ...}

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    for r in manifest.get('revisions', []):
        prev_data[r['rev']] = r
    print(f'Loaded previous manifest: {len(prev_data)} revisions')
else:
    print(f'No previous manifest found at {manifest_path}')

current_hashes = CURRENT_HASHES[TEST_SHEET]
ods_dir = DRIVE_DIR / TEST_SHEET


def read_drive_ods_hash(rev_num):
    """Read an existing .ods.gz from Drive, decompress, extract
    content.xml, return its SHA256. Returns None if file missing."""
    gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
    if not gz_path.exists():
        return None
    try:
        ods_bytes = gzip.decompress(gz_path.read_bytes())
        return ods_content_hash(ods_bytes)
    except Exception as e:
        print(f'  [warn] rev-{rev_num} decompress error: {e}')
        return None


# Spot-check: pick a few revisions from the test plan that have
# manifest data, and verify the manifest hash matches the actual file
print(f'\nSpot-checking manifest hashes against actual Drive files...')
spot_checks = [(rev, grp) for rev, grp in TEST_PLAN if rev in prev_data][:8]
spot_ok = 0
spot_mismatch = 0

for rev_num, group in spot_checks:
    manifest_hash = prev_data[rev_num].get('content_hash')
    drive_hash = read_drive_ods_hash(rev_num)

    if drive_hash is None:
        print(f'  rev-{rev_num:>5}: no .ods.gz on Drive (skipped)')
        continue

    if manifest_hash == drive_hash:
        spot_ok += 1
        tag = 'CURRENT' if manifest_hash in current_hashes else 'histor.'
        print(f'  rev-{rev_num:>5}: MATCH  manifest={manifest_hash[:16]}... '
              f'drive={drive_hash[:16]}... [{tag}]')
    else:
        spot_mismatch += 1
        print(f'  rev-{rev_num:>5}: MISMATCH!')
        print(f'           manifest: {manifest_hash[:32]}...')
        print(f'           drive:    {drive_hash[:32]}...')

print(f'\nSpot-check: {spot_ok} OK, {spot_mismatch} mismatch')
if spot_mismatch > 0:
    print('  WARNING: manifest data does not match Drive files!')
    print('  Using Drive files as ground truth for comparison.')
USE_DRIVE_FILES = True  # always compare against actual files when available

## Step 4: Run the Test

Downloads each revision in the test plan with **adaptive backoff**:

### Rate limiting
- **429 retry**: waits `current_delay + 4s` (always 4s above base rate)
- **After 429**: base delay **+1s**, speed-up threshold **+1**

### Speed-up probing
- After N consecutive successes (starts at **12**): decrease delay by **0.5s**
- Next probe after **N-1** successes, then **N-2**, etc. (gets more aggressive)
- On 429: threshold increases by 1 again (backs off)
- Minimum delay: **2s**

### Checkpoint file
- `checkpoint-{sheet}.json` on Drive — updated after every request
- Contains: last rev, delay, threshold, streak — everything to resume cleanly
- To restart from a specific revision: edit the file manually
- To start completely fresh: delete the file

### Canary
Stops if `CANARY_THRESHOLD` consecutive revisions return the same content hash.

In [ ]:
sheet_id = SHEETS[TEST_SHEET]
ods_dir.mkdir(exist_ok=True)

# ── Checkpoint file — the single source of "where were we?" ──
# Edit this file on Drive to resume from a specific revision,
# or delete it to start fresh.
checkpoint_path = DRIVE_DIR / f'checkpoint-{TEST_SHEET}.json'

# Results file — full log, written after every request
results_path = DRIVE_DIR / f'slow-validation-{TEST_SHEET}.json'

# ── Adaptive backoff state ──
MIN_DELAY = 2.0
RETRY_GAP = 4.0  # 429 retry wait = current_delay + this
current_delay = REQUEST_DELAY
ok_streak = 0              # consecutive successes without 429
speedup_threshold = 12     # successes needed before probing down
total_429s = 0
rate_changes = []          # log of all rate adjustments
resume_after_rev = None    # skip revisions up to and including this

# ── Load checkpoint if available ──
if checkpoint_path.exists():
    cp = json.loads(checkpoint_path.read_text())
    resume_after_rev = cp.get('last_rev')
    current_delay = cp.get('delay', current_delay)
    speedup_threshold = cp.get('speedup_threshold', speedup_threshold)
    ok_streak = cp.get('ok_streak', 0)
    total_429s = cp.get('total_429s', 0)
    print(f'Checkpoint loaded: resume after rev {resume_after_rev}, '
          f'delay={current_delay:.1f}s, speedup_threshold={speedup_threshold}, '
          f'ok_streak={ok_streak}')
else:
    print(f'No checkpoint — starting fresh')


def save_checkpoint(rev_num):
    """Write minimal checkpoint to Drive after every request."""
    checkpoint_path.write_text(json.dumps({
        'sheet': TEST_SHEET,
        'last_rev': rev_num,
        'delay': round(current_delay, 1),
        'speedup_threshold': speedup_threshold,
        'ok_streak': ok_streak,
        'total_429s': total_429s,
        'timestamp': now_iso(),
    }, indent=2))


# ── Load or init results ──
if results_path.exists():
    results = json.loads(results_path.read_text())
    done_revs = {r['rev'] for r in results.get('tests', [])}
    print(f'Results file: {len(done_revs)} revisions already logged')
else:
    results = {
        'sheet_key': TEST_SHEET,
        'sheet_id': sheet_id,
        'config': {
            'request_delay': REQUEST_DELAY,
            'canary_threshold': CANARY_THRESHOLD,
        },
        'started': now_iso(),
        'tests': [],
        'summary': {},
    }
    done_revs = set()


def save_results():
    """Write full results to Drive."""
    results['last_updated'] = now_iso()
    tests = results['tests']
    results['summary'] = {
        'total_tested': len(tests),
        'total_ok': sum(1 for t in tests if t['status'] == 'ok'),
        'total_error': sum(1 for t in tests if t['status'] == 'error'),
        'historical': sum(1 for t in tests if t.get('is_historical')),
        'current': sum(1 for t in tests if t.get('is_current')),
        'changed_from_prev': sum(1 for t in tests if t.get('changed_from_prev')),
        'replaced_on_drive': sum(1 for t in tests if t.get('replaced_on_drive')),
        'canary_triggered': results.get('canary_triggered', False),
    }
    results_path.write_text(json.dumps(results, indent=2))


def log_rate_change(reason, old_delay, new_delay):
    """Print and record a rate change."""
    arrow = '▲' if new_delay > old_delay else '▼'
    msg = (f'  {arrow} RATE {old_delay:.1f}s -> {new_delay:.1f}s  '
           f'(retry={new_delay + RETRY_GAP:.0f}s)  ({reason})')
    print(msg)
    rate_changes.append({
        'timestamp': now_iso(),
        'reason': reason,
        'old_delay': old_delay,
        'new_delay': new_delay,
        'retry_wait': new_delay + RETRY_GAP,
        'speedup_threshold': speedup_threshold,
    })


# ── Determine which revisions to skip (checkpoint-based) ──
skip_up_to = resume_after_rev is not None
revs_skipped = 0

# ── Main test loop ──
print(f'\n{"="*70}')
print(f'Starting: {TEST_SHEET} ({len(TEST_PLAN)} revisions)')
print(f'  delay={current_delay:.1f}s, retry_wait={current_delay + RETRY_GAP:.0f}s, '
      f'speedup_after={speedup_threshold}')
print(f'  Checkpoint: {checkpoint_path}')
print(f'  Results:    {results_path}')
print(f'{"="*70}\n')

consecutive_same = 0
last_hash = None
canary_triggered = False

for i, (rev_num, group) in enumerate(TEST_PLAN):
    # ── Skip logic: checkpoint-based resume ──
    if skip_up_to:
        if rev_num == resume_after_rev:
            skip_up_to = False  # found it, start from NEXT
            revs_skipped += 1
            continue
        else:
            revs_skipped += 1
            continue

    # ── Skip logic: already in results (from previous partial run) ──
    if rev_num in done_revs:
        print(f'  [{i+1}/{len(TEST_PLAN)}] rev-{rev_num:>5} ({group}): '
              f'skip (in results)')
        continue

    # ── 1. Download from API ──
    t0 = time.time()
    retry_wait = current_delay + RETRY_GAP
    print(f'  [{i+1}/{len(TEST_PLAN)}] rev-{rev_num:>5} ({group:>8}): ',
          end='', flush=True)

    ods_data, http_status, error, hit_429 = export_revision_ods(
        sheet_id, rev_num, retry_wait=retry_wait)
    dl_elapsed = time.time() - t0

    # ── Adaptive backoff: adjust on 429 ──
    if hit_429:
        total_429s += 1
        ok_streak = 0
        old_delay = current_delay
        current_delay += 1.0
        speedup_threshold += 1
        if current_delay != old_delay:
            log_rate_change(
                f'429 #{total_429s}, threshold->{speedup_threshold}',
                old_delay, current_delay)

    if not ods_data:
        print(f'ERROR (HTTP {http_status}: {error}) [{dl_elapsed:.1f}s]')
        results['tests'].append({
            'rev': rev_num,
            'group': group,
            'status': 'error',
            'http_status': http_status,
            'error': error,
            'hit_429': hit_429,
            'delay_at_time': current_delay,
            'timestamp': now_iso(),
            'elapsed': round(dl_elapsed, 2),
        })
        consecutive_same = 0
        last_hash = None
        save_results()
        save_checkpoint(rev_num)
        time.sleep(current_delay)
        continue

    # ── Adaptive backoff: count successes, probe lower delay ──
    if not hit_429:
        ok_streak += 1
        if ok_streak >= speedup_threshold:
            old_delay = current_delay
            current_delay = max(current_delay - 0.5, MIN_DELAY)
            ok_streak = 0
            # Next probe comes sooner (threshold decreases)
            speedup_threshold = max(speedup_threshold - 1, 6)
            if current_delay != old_delay:
                log_rate_change(
                    f'{speedup_threshold + 1} OK, next probe after '
                    f'{speedup_threshold}',
                    old_delay, current_delay)
            else:
                # Already at MIN_DELAY, reset streak but don't log
                pass

    # ── 2. Hash the new download ──
    content_hash = ods_content_hash(ods_data)
    ods_size = len(ods_data)
    is_current = content_hash in current_hashes
    is_historical = not is_current and content_hash is not None

    # ── 3. Read existing file from Drive for comparison ──
    drive_hash = read_drive_ods_hash(rev_num)
    manifest_hash = prev_data.get(rev_num, {}).get('content_hash')
    prev_hash = drive_hash or manifest_hash
    prev_source = 'drive' if drive_hash else ('manifest' if manifest_hash else None)

    changed_from_prev = (prev_hash is not None and content_hash != prev_hash)
    prev_was_current = prev_hash in current_hashes if prev_hash else None

    # ── 4. Canary check ──
    if content_hash == last_hash:
        consecutive_same += 1
    else:
        consecutive_same = 1
        last_hash = content_hash

    # ── 5. Build result entry ──
    entry = {
        'rev': rev_num,
        'group': group,
        'status': 'ok',
        'content_hash': content_hash,
        'ods_size': ods_size,
        'is_current': is_current,
        'is_historical': is_historical,
        'prev_hash': prev_hash[:24] + '...' if prev_hash else None,
        'prev_source': prev_source,
        'prev_was_current': prev_was_current,
        'changed_from_prev': changed_from_prev,
        'consecutive_same': consecutive_same,
        'hit_429': hit_429,
        'delay_at_time': current_delay,
        'speedup_threshold': speedup_threshold,
        'ok_streak': ok_streak,
        'timestamp': now_iso(),
        'elapsed': round(dl_elapsed, 2),
        'replaced_on_drive': False,
    }

    # ── 6. Print status ──
    parts = [f'{ods_size:,}b']
    if is_current:
        parts.append('CURRENT')
    else:
        parts.append(f'HISTORICAL ({content_hash[:12]}...)')

    if changed_from_prev:
        if prev_was_current and is_historical:
            parts.append('RECOVERED!')
        elif not prev_was_current and is_current:
            parts.append('REGRESSED')
        else:
            parts.append('CHANGED')
    elif prev_hash:
        parts.append(f'same as prev ({prev_source})')
    parts.append(f'[{dl_elapsed:.1f}s]')
    if consecutive_same > 1:
        parts.append(f'(same x{consecutive_same})')
    parts.append(f'd={current_delay:.1f}s')
    parts.append(f'ok={ok_streak}/{speedup_threshold}')
    print(' '.join(parts))

    # ── 7. Replace on Drive if recovered ──
    if changed_from_prev and is_historical and prev_was_current:
        gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
        gz_data = gzip.compress(ods_data, compresslevel=6)
        gz_path.write_bytes(gz_data)
        entry['replaced_on_drive'] = True
        print(f'         >>> REPLACED on Drive: {gz_path.name} '
              f'({len(gz_data):,} gz bytes)')

    results['tests'].append(entry)
    done_revs.add(rev_num)
    save_results()
    save_checkpoint(rev_num)

    # ── 8. Canary: stop if collapsed ──
    if consecutive_same >= CANARY_THRESHOLD:
        print(f'\n  *** CANARY: {consecutive_same} identical hashes in a row ***')
        print(f'  *** Hash: {content_hash[:32]}...')
        print(f'  *** Method C has collapsed to current content. Stopping.')
        results['canary_triggered'] = True
        results['canary_at_rev'] = rev_num
        results['canary_after_n'] = i + 1
        save_results()
        canary_triggered = True
        break

    # ── 9. Wait (adaptive delay) ──
    remaining_delay = max(0, current_delay - (time.time() - t0))
    if remaining_delay > 0:
        time.sleep(remaining_delay)

# Final save
results['completed'] = now_iso()
results['adaptive_backoff'] = {
    'final_delay': current_delay,
    'total_429s': total_429s,
    'final_speedup_threshold': speedup_threshold,
    'rate_changes': rate_changes,
}
save_results()

if revs_skipped:
    print(f'\n  Skipped {revs_skipped} revisions (checkpoint: rev {resume_after_rev})')
if not canary_triggered:
    print(f'\n  All {len(TEST_PLAN)} revisions tested without canary trigger!')
print(f'  Final: delay={current_delay:.1f}s, retry={current_delay + RETRY_GAP:.0f}s, '
      f'speedup_threshold={speedup_threshold}')
print(f'  Total 429s: {total_429s}')

## Step 5: Analyze Results

In [ ]:
# Load and analyze results
results = json.loads(results_path.read_text())
tests = results['tests']

print(f'{"="*70}')
print(f'RESULTS: {TEST_SHEET}')
print(f'{"="*70}')
print(f'  Started:  {results.get("started", "?")}')
print(f'  Completed: {results.get("completed", "?")}')
print(f'  Initial delay: {results["config"]["request_delay"]}s')
print()

s = results.get('summary', {})
print(f'  Total tested:       {s.get("total_tested", 0)}')
print(f'  Successful (OK):    {s.get("total_ok", 0)}')
print(f'  Errors:             {s.get("total_error", 0)}')
print(f'  Historical:         {s.get("historical", 0)}')
print(f'  Current (broken):   {s.get("current", 0)}')
print(f'  Changed from prev:  {s.get("changed_from_prev", 0)}')
print(f'  Replaced on Drive:  {s.get("replaced_on_drive", 0)}')
print(f'  Canary triggered:   {s.get("canary_triggered", False)}')

# Adaptive backoff summary
ab = results.get('adaptive_backoff', {})
if ab:
    print(f'\n  Adaptive backoff:')
    print(f'    Final delay:       {ab.get("final_delay", "?")}s')
    print(f'    Final threshold:   {ab.get("final_speedup_threshold", "?")}')
    print(f'    Total 429s:        {ab.get("total_429s", 0)}')
    rcs = ab.get('rate_changes', [])
    if rcs:
        print(f'    Rate changes ({len(rcs)}):')
        for rc in rcs:
            arrow = '+' if rc['new_delay'] > rc['old_delay'] else '-'
            thr = f' thr={rc["speedup_threshold"]}' if 'speedup_threshold' in rc else ''
            print(f'      {arrow} {rc["old_delay"]:.1f}s -> {rc["new_delay"]:.1f}s  '
                  f'retry={rc.get("retry_wait", "?"):.0f}s{thr}  ({rc["reason"]})')

# Checkpoint status
if checkpoint_path.exists():
    cp = json.loads(checkpoint_path.read_text())
    print(f'\n  Checkpoint: rev {cp["last_rev"]}, delay={cp["delay"]}s, '
          f'threshold={cp["speedup_threshold"]}, streak={cp["ok_streak"]}')

# Show per-group breakdown
print(f'\n  Per-group results:')
for group in ['control', 'broken', 'boundary']:
    group_tests = [t for t in tests if t.get('group') == group and t['status'] == 'ok']
    if not group_tests:
        continue
    hist = sum(1 for t in group_tests if t.get('is_historical'))
    curr = sum(1 for t in group_tests if t.get('is_current'))
    changed = sum(1 for t in group_tests if t.get('changed_from_prev'))
    print(f'    {group:>8}: {len(group_tests)} tested, '
          f'{hist} historical, {curr} current, {changed} changed')

# Detailed results table
print(f'\n{"Rev":>6}  {"Group":>8}  {"Size":>8}  '
      f'{"Content":>8}  {"vs Prev":>10}  {"Delay":>5}  '
      f'{"ok/thr":>7}  {"Hash (first 16)"}')
print('-' * 100)
for t in tests:
    if t['status'] == 'error':
        flag = ' *429' if t.get('hit_429') else ''
        print(f'{t["rev"]:>6}  {t["group"]:>8}  {"ERROR":>8}  '
              f'HTTP {t.get("http_status", "?")}{flag}')
        continue
    content = 'CURRENT' if t.get('is_current') else 'histor.'
    vs_prev = ''
    if t.get('changed_from_prev'):
        if t.get('prev_was_current') and t.get('is_historical'):
            vs_prev = 'RECOVERED'
        elif not t.get('prev_was_current') and t.get('is_current'):
            vs_prev = 'regressed'
        else:
            vs_prev = 'changed'
    elif t.get('prev_hash'):
        vs_prev = 'same'
    else:
        vs_prev = 'new'
    h = t.get('content_hash', '?')[:16]
    delay = t.get('delay_at_time', '?')
    delay_str = f'{delay:.1f}s' if isinstance(delay, (int, float)) else str(delay)
    ok = t.get('ok_streak', '?')
    thr = t.get('speedup_threshold', '?')
    ok_thr = f'{ok}/{thr}'
    flag_429 = ' *' if t.get('hit_429') else ''
    print(f'{t["rev"]:>6}  {t["group"]:>8}  '
          f'{t.get("ods_size",0):>8,}  {content:>8}  '
          f'{vs_prev:>10}  {delay_str:>5}  '
          f'{ok_thr:>7}  {h}...{flag_429}')

In [ ]:
# === Key Question: Did pacing help? ===

broken_tests = [t for t in tests
                if t.get('group') == 'broken' and t['status'] == 'ok']
broken_historical = [t for t in broken_tests if t.get('is_historical')]
broken_current = [t for t in broken_tests if t.get('is_current')]

print(f'\n{"="*70}')
print(f'KEY FINDING: Did slower pacing recover historical data?')
print(f'{"="*70}')
print(f'  Previously-broken revisions tested:  {len(broken_tests)}')
print(f'  Now returning HISTORICAL content:    {len(broken_historical)}')
print(f'  Still returning CURRENT content:     {len(broken_current)}')
print()

if len(broken_historical) > 0:
    pct = 100 * len(broken_historical) / len(broken_tests)
    print(f'  >>> YES! {pct:.0f}% of previously-broken revisions now return '
          f'historical data!')
    print(f'  >>> Pacing hypothesis CONFIRMED.')
    print(f'\n  Recovered revisions:')
    for t in broken_historical:
        print(f'    rev-{t["rev"]:>5}: {t["ods_size"]:,} bytes, '
              f'hash={t["content_hash"][:20]}...')
elif len(broken_tests) == 0:
    print('  No broken revisions were tested (all errored out).')
else:
    print(f'  >>> NO — all previously-broken revisions still return current content.')
    print(f'  >>> Pacing alone does not fix Method C.')
    print(f'  >>> The collapse is not a rate-limiting issue.')

# Control group verification
control_tests = [t for t in tests
                 if t.get('group') in ('control', 'boundary') and t['status'] == 'ok']
control_same = sum(1 for t in control_tests if not t.get('changed_from_prev'))
control_changed = sum(1 for t in control_tests if t.get('changed_from_prev'))

print(f'\n  Control group ({len(control_tests)} revisions):')
print(f'    Same as previous run: {control_same}')
print(f'    Changed from previous: {control_changed}')
if control_same == len(control_tests):
    print(f'    >>> Control group is consistent — our hashes are reliable.')
elif control_changed > 0:
    print(f'    >>> WARNING: {control_changed} control revisions returned '
          f'different data! Method C is non-deterministic.')

---

## For Claude: How to check progress

When the user points you to this notebook and asks how it's going, here's
what to do. All state lives on Google Drive at
`/content/drive/MyDrive/ubl-gc-revisions/`.

### 1. Check the checkpoint file (quick status)

The checkpoint is a tiny JSON file updated after every single request.

**For ubl25_library:**
```
checkpoint-ubl25_library.json
```
**For ubl25_documents:**
```
checkpoint-ubl25_documents.json
```

Read it. It looks like:
```json
{
  "sheet": "ubl25_library",
  "last_rev": 847,
  "delay": 4.5,
  "speedup_threshold": 10,
  "ok_streak": 7,
  "total_429s": 3,
  "timestamp": "2026-02-17T14:23:01.123456Z"
}
```

**What to report:**
- `last_rev` / MAX_REV = **progress** (e.g. 847/2005 = 42%)
- `delay` = current adaptive rate (started at 5.0s)
- `total_429s` = how many rate-limit hits so far
- `timestamp` = when the last request happened (is it still running?)

### 2. Check the results file (detailed analysis)

```
slow-validation-ubl25_library.json
```

The `summary` section has aggregate counts:
- `total_tested`, `total_ok`, `total_error`
- `historical` vs `current` — how many returned real historical data
- `replaced_on_drive` — how many improved over previous downloads

The `adaptive_backoff` section (written at completion) has:
- `final_delay`, `total_429s`, `rate_changes` log

### 3. Check the downloaded .ods.gz files

```python
import os
ods_dir = '/content/drive/MyDrive/ubl-gc-revisions/ubl25_library/'
files = sorted(os.listdir(ods_dir))
print(f'{len(files)} files downloaded')
print(f'First: {files[0]}, Last: {files[-1]}')
```

### 4. Is it still running?

Compare the checkpoint `timestamp` to current time:
- **< 30s ago**: actively running
- **1-10 min ago**: might be stuck on a retry or Colab disconnected
- **> 10 min ago**: likely stopped (Colab timeout, crash, or finished)

If stopped mid-run: just re-run cell 7 (Step 4). The checkpoint will
resume from `last_rev` with the saved adaptive state.

### 5. Sheet details

| Sheet key | Google Sheet ID | Max Rev |
|-----------|----------------|---------|
| `ubl25_library` | `18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY` | 2005 |
| `ubl25_documents` | `1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg` | 2204 |

### 6. When it's done

Both sheets fully downloaded = **2005 + 2204 = 4209 revisions** total.
Each stored as `rev-{N}.ods.gz` on Drive. The results JSON has the
content hash for every revision, showing which are genuinely historical
vs which returned current-version content.